# AI Detector — full MuRIL fine-tuning on a Colab GPU

**Choose Runtime → Change runtime type → T4 GPU (or another available GPU), then Runtime → Run all.** Authorize the Google Drive prompt. No API key is required.

This trains the embeddings and **all 12 MuRIL transformer blocks**, compares three learning rates using **five source-grouped folds**, and evaluates twelve hybrid classifier configurations. Early stopping has its own grouped validation subset inside each training fold. The existing test split does not select the model.

The notebook runs the complete search, verifies serving parity, and exports the selected models. It does **not** guarantee an accuracy improvement. Frozen MuRIL remains eligible to win.

Completed folds and final models persist in Drive; frequent epoch checkpoints stay on Colab's local disk to avoid Drive Bin buildup. Keep several GB free for final models and exports, plus space for any older saved checkpoints. After a runtime reset, the unfinished fold/refit may need to repeat; completed folds are reused. Keep the same v2 run folder. Colab GPU availability and session length depend on your account. **Use only one runtime per run folder.** No anti-idle or quota-bypass code is used.

This notebook needs your Google sign-in to allocate a GPU and mount Drive. Preparing the notebook does not mean a GPU run has already executed.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU attached. Select Runtime > Change runtime type > T4 GPU, reconnect, then Run all.")
print("Colab GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

# Keep this name unchanged when resuming. Use a NEW name for a different experiment.
RUN_ROOT = Path('/content/drive/MyDrive/AI-Detector/full-muril-gpu-v2-scipy114')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
REPO = Path('/content/AI-Detector-full-muril')
# Immutable source version keeps resumptions reproducible.
SOURCE_COMMIT = 'b9cf2495bfccf3008676ce052f6539b313328d59'
REPO_URL = 'https://github.com/parthpathak719/AI-Detector.git'

def run(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)

if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f"Refusing to replace a non-repository directory: {REPO}")
    run(['git', 'clone', '--no-checkout', REPO_URL, REPO])
run(['git', '-C', REPO, 'fetch', 'origin', SOURCE_COMMIT])
run(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT])
os.chdir(REPO)
os.environ['HF_HUB_DISABLE_XET'] = '1'
print('Persistent experiment directory:', RUN_ROOT)

## Install an isolated Python 3.12 CUDA environment

This uses uv to provision Python 3.12 independently of Colab's system Python and its optional venv/ensurepip components. It installs CUDA-enabled PyTorch in the isolated environment; no notebook-kernel restart is needed.


In [ ]:
# Colab's system Python may lack ensurepip/venv, or may be Python 3.13.
# Use uv to provision a separate Python 3.12 interpreter and seed pip.
import time
ENVIRONMENT = REPO / '.colab-py312'
PYTHON = ENVIRONMENT / 'bin/python'
probe = subprocess.run(
    [str(PYTHON), '-c', 'import sys, pip; assert sys.version_info[:2] == (3, 12)'],
    capture_output=True, text=True,
) if PYTHON.exists() else None
if probe is None or probe.returncode:
    if ENVIRONMENT.exists():
        backup = ENVIRONMENT.with_name('.colab-py312-incomplete-' + str(time.time_ns()))
        ENVIRONMENT.rename(backup)
        print('Preserved incomplete environment at', backup)
    run([sys.executable, '-m', 'pip', 'install', 'uv==0.7.22'])
    run([sys.executable, '-m', 'uv', 'python', 'install', '3.12'])
    run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', ENVIRONMENT])
run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
run([PYTHON, '-m', 'pip', 'install', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'])
run([PYTHON, '-m', 'pip', 'install', '-r', 'requirements.txt', 'gdown==5.2.0'])
# Compatibility workaround for the observed SciPy 1.15.3 L-BFGS native crash.
run([PYTHON, '-m', 'pip', 'install', '--no-deps', 'scipy==1.14.1'])
run([PYTHON, '-c', "import torch; assert torch.cuda.is_available(), 'CUDA unavailable in training environment'; print(torch.__version__, torch.cuda.get_device_name(0))"])


## Import your supplied project data and download MuRIL

The default archive is the Drive link you supplied. The importer extracts only strict-cleaned JSONL data and embedding arrays; it does not execute notebook/archive code or load legacy pickles.

If that sharing link stops working, put the same archive in your Drive and change `ARCHIVE` below to its mounted path. Keep the original cleaned dataset unchanged to preserve the comparison.

In [ ]:
ARCHIVE = REPO / '.cache/source/project-progress.zip'
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
if not ARCHIVE.exists():
    from zipfile import is_zipfile
    partial = ARCHIVE.with_suffix('.zip.part')
    run([PYTHON, '-m', 'gdown', 'https://drive.google.com/uc?id=11ueCWmhonSNsHO1FCjsvWBQscwdzNxbH', '-O', partial])
    if not is_zipfile(partial):
        raise RuntimeError('Download is not a ZIP archive. Check the Drive sharing link or set ARCHIVE to your mounted Drive file.')
    partial.replace(ARCHIVE)
run([PYTHON, '-m', 'scripts.import_data', ARCHIVE])
run([PYTHON, '-m', 'scripts.download_muril'])
# Exercise the exact head-initialization path before the long GPU job.
run([PYTHON, '-X', 'faulthandler', '-u', '-m', 'scripts.check_colab_optimizer'])


## Run or resume the complete five-fold search

This compatibility run recomputes all 15 frozen baseline folds under SciPy 1.14.1 before GPU fine-tuning, using the same source groups and settings. This keeps frozen and fine-tuned candidates on the same numerical-library version. The original CPU results and failed GPU-v1 attempt remain separate. It checks gradients in every transformer block and the embeddings.

Completed fold reports and logs live in Drive. Epoch checkpoints live on local disk. On a fresh runtime, the launcher restores an older Drive checkpoint for the unfinished fold if available, without changing the original. Otherwise the unfinished fold/refit must replay. This storage update preserves the v2 experiment and does not change its training protocol. Full precision is used; TF32 is disabled. **This cell may run for hours.** Leave it running; no manual action is needed between folds or tasks. If it fails, read the final error and `training.log`; do not delete checkpoints to hide the error.

In [ ]:
command = [str(PYTHON), '-X', 'faulthandler', '-u', '-m', 'scripts.colab_run', '--run-root', str(RUN_ROOT), '--recompute-baselines']
with (RUN_ROOT / 'training.log').open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = process.wait()
        if returncode == -11:
            raise RuntimeError(
                "Training hit a native SIGSEGV crash. The faulthandler traceback above "
                f"and in {RUN_ROOT / 'training.log'} identifies the active Python frames. "
                "Keep the checkpoints and share that traceback to diagnose the failing library."
            )
        if returncode:
            raise RuntimeError(f"Training stopped with exit code {returncode}. See {RUN_ROOT / 'training.log'}; saved checkpoints remain available.")
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise

## Export the selected models and results

This cell runs only after the whole experiment succeeds, including application-loader parity checks. It refuses to export a partial run.

- **`results.zip`**: small evaluation reports to share for review; no training passages or optimizer states.
- **`selected_models.zip`**: the selected application artifacts, including full MuRIL checkpoints only where they won. This can be several GB; download it from Drive.
- **`results.md`**: comparison table with frozen/full CV scores and selected test accuracy.

All files are saved in **My Drive → AI-Detector → full-muril-gpu-v2-scipy114**. Do not upload large encoder weights to normal Git.

In [ ]:
run([PYTHON, '-m', 'scripts.export_full_run', '--run-root', RUN_ROOT])
from IPython.display import display, Markdown
display(Markdown((RUN_ROOT / 'results.md').read_text(encoding='utf-8')))
print('Results:', RUN_ROOT / 'results.zip')
print('Selected application models:', RUN_ROOT / 'selected_models.zip')

## Use the selected models on Windows

Download `selected_models.zip` from the Drive folder. Stop your local app with **Ctrl+C**, then run the following in your project directory. Back up `artifacts` before extraction. The ZIP contains an `artifacts/` folder and excludes the private training data.

```powershell
git pull
Rename-Item artifacts artifacts-before-full-finetune
Expand-Archive -LiteralPath "$HOME\Downloads\selected_models.zip" -DestinationPath .
.\.venv\Scripts\python.exe -m scripts.verify_artifacts
.\.venv\Scripts\python.exe -m uvicorn detector.app:app --host 127.0.0.1 --port 8000
```

If the backup folder already exists, choose a new backup name. Full models use more RAM/disk than the frozen classifier artifacts. These results remain short-text task evaluations; they do not establish document-level authorship accuracy.